# Last compute step: quantization and latency for both architectures

`stage_quantize` used to begin with `arch = args.arch[0]`, so passing
`--arch compact full` quantized only `compact` and said nothing about it.
`quantize.csv` and `latency.csv` therefore cover the 0.27M model alone. That
is fixed; this notebook regenerates both architectures. See docs/BUGS.md
item 17.

About 10 minutes. Nothing else in the study needs a GPU after this.

**Inputs to attach:**

| Input | Why |
|---|---|
| `tawsifurrahman/tuberculosis-tb-chest-xray-dataset` | regenerates the splits, and rebuilds the cache if it is not restored |
| `wenhaolu49/notebook7981e10859` | the trained checkpoints and the image cache |

The Montgomery/Shenzhen dataset is **not** needed here; nothing in this
notebook touches the external cohorts.

Use `notebook7981e10859` rather than `notebook8807a976de`: the later run
trained `full` for 30 epochs under the old naming scheme and overwrote the
10-epoch `full_faithful_*.pth` files, so its checkpoints disagree with the
10-epoch rows in `baseline.csv`.

**GPU T4** (either count), **Internet on**.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/AIscend-Research/lightweight-tb-net"
REPO_DIR = "/kaggle/working/lightweight-tb-net"
SEEDS    = [0, 1, 2, 3, 4]
PRIOR_RUNS = ["/kaggle/input/notebooks/wenhaolu49/notebook7981e10859",
              "/kaggle/input/notebooks/wenhaolu49/notebook8807a976de"]

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif not ON_KAGGLE:
    REPO_DIR = os.getcwd()
os.chdir(REPO_DIR)
SRC = os.path.join(REPO_DIR, "src")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                        text=True).stdout.strip()
print("commit:", COMMIT)

# onnxruntime for the benchmark, onnxscript because torch 2.6+ routes
# torch.onnx.export through the dynamo exporter by default.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "onnx", "onnxruntime", "onnxscript"], check=True)
import onnxruntime
print("onnxruntime", onnxruntime.__version__)


def run(*cmd, check=True):
    cmd = [sys.executable] + list(cmd)
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        print(line, end="")
    p.wait()
    print(f"[{(time.time() - t0) / 60:.1f} min, exit {p.returncode}]")
    if check and p.returncode != 0:
        raise RuntimeError("".join(tail[-40:]))

In [ ]:
def find_dir(root, must_contain):
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        if all(m in dirnames for m in must_contain):
            return dirpath
    return None


def restore(src, dst):
    if src and os.path.isdir(src):
        shutil.copytree(src, os.path.join(REPO_DIR, dst), dirs_exist_ok=True)
        print(f"restored {dst}/ from {src}")
        return True
    return False


DATA_PATH = (find_dir("/kaggle/input", ["Normal", "Tuberculosis"])
             if ON_KAGGLE else os.path.join(REPO_DIR, "data"))
print("DATA_PATH =", DATA_PATH)

cands = [f"{r}/{s}" for r in PRIOR_RUNS
         for s in ("artifacts/checkpoints", "lightweight-tb-net/checkpoints")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoints", recursive=True))
restore(next((d for d in cands if os.path.isdir(d)), None), "checkpoints")

ccands = [f"{r}/lightweight-tb-net/cache" for r in PRIOR_RUNS]
ccands += sorted(glob.glob("/kaggle/input/**/cache", recursive=True))
restore(next((d for d in ccands if os.path.isdir(d)), None), "cache")

n_ckpt = len(glob.glob(f"{REPO_DIR}/checkpoints/*.pth"))
have_cache = os.path.exists(f"{REPO_DIR}/cache/faithful.npy")
print(f"checkpoints: {n_ckpt} | cache: {have_cache}")
assert n_ckpt, ("No checkpoints. Attach wenhaolu49/notebook7981e10859 as a "
                "notebook-output input; this notebook does not retrain.")

if DATA_PATH:
    run(f"{SRC}/make_splits.py", "--data-path", DATA_PATH,
        "--out", f"{REPO_DIR}/data_splits", "--seed", "42")
if not have_cache:
    run(f"{SRC}/build_cache.py", "--data-path", DATA_PATH, "--variant", "faithful")

In [ ]:
# Clear both files first: the compact rows already in the repo copy would
# otherwise be duplicated by this rerun.
for f in ("quantize.csv", "latency.csv", "latency_failures.csv"):
    p = f"{REPO_DIR}/results/{f}"
    if os.path.exists(p):
        os.remove(p)
        print("cleared results/" + f)

run(f"{SRC}/experiments.py", "--stage", "quantize", "--force",
    "--arch", "compact", "full", "--caches", "faithful",
    "--seeds", *map(str, SEEDS))

import pandas as pd
for f in ("quantize.csv", "latency.csv", "latency_failures.csv"):
    p = f"{REPO_DIR}/results/{f}"
    print(f"\n--- {f} ---")
    if os.path.exists(p):
        d = pd.read_csv(p)
        keys = [k for k in ("arch", "base", "quant") if k in d.columns]
        cols = [c for c in ("acc", "sens", "size_mb", "lat_ms_median")
                if c in d.columns]
        display(d.groupby(keys)[cols].agg(["mean", "std"]).round(3))
    else:
        print("absent")

# Both architectures must appear now; that was the whole point.
q = pd.read_csv(f"{REPO_DIR}/results/quantize.csv")
print("\narchitectures in quantize.csv:", sorted(q["arch"].unique()))

In [ ]:
for f in glob.glob(f"{REPO_DIR}/results/summary_*.csv"):
    os.remove(f)
run(f"{SRC}/experiments.py", "--stage", "summary", check=False)
run(f"{SRC}/analyze.py")
run(f"{SRC}/make_figures.py", check=False)

OUT = "/kaggle/working/for_repo"
shutil.rmtree(OUT, ignore_errors=True)
for sub in ("results", "figures"):
    shutil.copytree(os.path.join(REPO_DIR, sub), os.path.join(OUT, sub))
shutil.make_archive(OUT, "zip", OUT)
print(f"\n-> for_repo.zip ({os.path.getsize(OUT + '.zip') / 1024 ** 2:.1f} MB)")